In [ ]:
import os
from pathlib import Path
import pandas as pd 
import matplotlib.pyplot as plt
import math
from statistics import mean 
import pdb
from scipy.stats import ttest_ind
import json
import seaborn as sns
import matplotlib.ticker as ticker
import numpy as np

# 1) Resolve the folder this notebook lives in
try:
    # .py script
    _here = Path(__file__).resolve().parent
except NameError:
    import sys
    main_mod = sys.modules.get('__main__')
    if getattr(main_mod, '__file__', None):
        _here = Path(main_mod.__file__).resolve().parent
    else:
        _here = Path.cwd()

# 2) Look for a sibling 'data' folder here or up to 3 parents up
_candidates = [
    _here / "data",
    _here.parent / "data",
    _here.parent.parent / "data",
    _here.parent.parent.parent / "data",
]
data_dir_path = next((p for p in _candidates if p.is_dir()), None)

# 3) If none found, allow CSVs to live right next to the notebook/script
if data_dir_path is None:
    data_dir_path = _here

repo_root_path = data_dir_path.parent if data_dir_path.name.lower() == "data" else _here
repo_root = str(repo_root_path)  # keep as str for os.path.join compatibility
directory = str(data_dir_path)

print(f"Repository root set: {repo_root}")
print(f"Data directory set: {directory}")

# 4) Read sub*.csv from the chosen data directory 
dataframes = []
for p in sorted(Path(directory).iterdir()):
    if p.is_file() and p.name.startswith("sub") and p.suffix.lower() == ".csv":
        print(f"Reading: {p.name}")
        dataframes.append(pd.read_csv(p))
    else:
        print(f"Skipping: {p.name}")

# 5) Preview
for i, df in enumerate(dataframes, start=1):
    print(f"\nFirst few rows of DataFrame {i}:")
    print(df.head())



In [ ]:
# Set directory to the data folder within the repository.
data_dir = os.path.join(repo_root, "data")
os.chdir(data_dir)

# Load data
CrypticCreatures = pd.read_csv("Table_CrypticCreatures_YaleCohort.csv")
print(CrypticCreatures.columns)
CrypticCreatures = CrypticCreatures.sort_values(by=['id', 'task_id', 'run', 'trial']).reset_index(drop=True)

# Define the color palette for all plots
colors_palette = ['#775D0F', '#410016', '#B89C33', '#E15304', '#6E540A','#D95A3B', '#9FA165', '#503345', '#D3CFC7']


Below is the implementation of an optimal Bayesian observer

In [ ]:
def bayesian_update(
    prior_beliefs: dict,
    chosen_features: dict,                 # {dim: value} for CHOSEN stimulus (only differing dims)
    unchosen_stimuli_features: dict,       # {stim_id: {dim: value}} for UNCHOSEN (only differing dims)
    feedback: int,                         # 1 = rewarded, 0 = not rewarded
    differing_dimensions: list,            # list of dims that differ on this trial
    task_id,                               # kept for API parity (unused)
    epsilon: float = 0.0,                
):
    """
    Feature-based (token-based) likelihood:

    If feedback == 1 (rewarded):
        Informative set Uc = { (d, v_c) on the chosen stimulus | v_c NOT present on ANY unchosen at d }
        k = |Uc|
        L(token) = 1/k for token in Uc; all other tokens get epsilon
        If k == 0 → non-informative: all tokens get epsilon

    If feedback == 0 (not rewarded):
        Informative set Uu = { (d, v) that appear on UNCHOSEN at d AND v != v_c (chosen value at d) }
        k = |Uu|
        L(token) = 1/k for token in Uu; chosen tokens remain at epsilon
        If k == 0 → non-informative: all tokens get epsilon
    """

    # --- Collect values appearing on UNCHOSEN, per dimension (restricted to differing dims) ---
    vals_u_by_dim = {}
    for uns in unchosen_stimuli_features.values():
        for d, v in uns.items():
            if d not in differing_dimensions or v is None:
                continue
            vals_u_by_dim.setdefault(d, set()).add(v)

    # Initialize all tokens' likelihoods to epsilon
    likelihoods = {tok: epsilon for tok in prior_beliefs.keys()}

    if feedback == 1:
        # ---- Rewarded: unique chosen tokens (present on chosen, absent on all unchosen at same dim) ----
        unique_chosen = set()
        for d in differing_dimensions:
            v_c = chosen_features.get(d, None)
            if v_c is None:
                continue
            if v_c not in vals_u_by_dim.get(d, set()):
                unique_chosen.add((d, v_c))

        k = len(unique_chosen)
        if k > 0:
            share = 1.0 / k
            for tok in unique_chosen:
                if tok in likelihoods:
                    likelihoods[tok] = share

    else:
        # ---- Not rewarded: informative unchosen tokens that differ from chosen at that dim ----
        informative = set()
        for d in differing_dimensions:
            v_c = chosen_features.get(d, None)
            for v in vals_u_by_dim.get(d, set()):
                if v is None:
                    continue
                if v != v_c:
                    informative.add((d, v))

        k = len(informative)
        if k > 0:
            share = 1.0 / k
            for tok in informative:
                if tok in likelihoods:
                    likelihoods[tok] = share
        # chosen tokens remain at epsilon

    # --- Bayes rule over the full hypothesis set in prior_beliefs ---
    unnormalized = {t: likelihoods[t] * prior_beliefs[t] for t in prior_beliefs}
    Z = sum(unnormalized.values())

    # Fallback if everything zero (can happen if epsilon == 0 and no informative tokens)
    if Z == 0.0:
        n = len(prior_beliefs) if prior_beliefs else 1
        posterior_beliefs = {t: 1.0 / n for t in prior_beliefs}
    else:
        posterior_beliefs = {t: unnormalized[t] / Z for t in prior_beliefs}

    return posterior_beliefs


In [ ]:
def compute_entropy(beliefs, scaling_factor=100):
    """
    Compute entropy and BLR (Bayesian Learner Confidence).
    
    beliefs: Dictionary of posterior probabilities for each feature.
    scaling_factor: Lambda for scaling confidence (default 1.0).
    
    Returns:
    - entropy: The entropy of the belief distribution.
    - BLR_confidence: The optimal confidence derived from entropy.
    """
    # Convert belief values to a numpy array of probabilities
    probs = np.array(list(beliefs.values()))
    
    if len(probs) == 0:
        return 0.0, 0.0  # Return 0 for both entropy and confidence if no probs
    
    # Compute entropy
    entropy = -np.sum(probs * np.log2(probs + 1e-9))  # Add a small value to avoid log(0)
    
    # Max possible entropy: when all hypotheses are equally likely
    max_entropy = np.log2(len(probs))  # N = number of possible hypotheses/features
    
    # Compute optimal Bayesian Learner Confidence
    if max_entropy != 0:
        BLR_confidence = scaling_factor * ((max_entropy - entropy) / max_entropy) 
    else:
        BLR_confidence = 0.0  # If max_entropy is zero, confidence is 0

    return entropy, BLR_confidence


Run it within and for all participants

In [ ]:
def process_participant(CrypticCreatures, id_tested, task_id_tested):
    # Filter the data for the given participant and task
    participant_data = CrypticCreatures[
        (CrypticCreatures['id'] == id_tested) & (CrypticCreatures['task_id'] == task_id_tested)
    ].reset_index(drop=True)

    # Merge 'IDs' and 'EDs' into a general shift variable
    participant_data['shifts'] = participant_data['IDs'] | participant_data['EDs']

    # Identify all stimulus columns
    all_columns = participant_data.columns
    stim_columns = [col for col in all_columns if col.startswith('stim')]

    # Exclude irrelevant dimensions
    irrelevant_dimensions = ['chosen', 'position', 'outcome']
    dimensions = sorted(set(col.split('_')[1] for col in stim_columns if '_' in col))
    dimensions = [dim for dim in dimensions if dim not in irrelevant_dimensions]

    # Identify the possible stimuli (e.g., stim1, stim2, stim3)
    possible_stimuli = sorted(set(col.split('_')[0] for col in stim_columns if '_' in col))

    # --- Pre-computation ---
    # stimuli_present
    participant_data['stimuli_present'] = [[] for _ in range(len(participant_data))]
    for idx, row in participant_data.iterrows():
        present = []
        for stim_id in possible_stimuli:
            if not pd.isna(row.get(f'{stim_id}_position')):
                present.append(stim_id)
        participant_data.at[idx, 'stimuli_present'] = present

    # stimuli_features
    participant_data['stimuli_features'] = [{} for _ in range(len(participant_data))]
    for idx, row in participant_data.iterrows():
        features_by_stim = {}
        for stim_id in participant_data.at[idx, 'stimuli_present']:
            stim_feats = {}
            for dim in dimensions:
                col_name = f'{stim_id}_{dim}'
                val = row.get(col_name)
                if pd.notna(val):
                    if isinstance(val, (int, float)):
                        stim_feats[dim] = str(int(val))
                    else:
                        stim_feats[dim] = str(val).strip()
            features_by_stim[stim_id] = stim_feats
        participant_data.at[idx, 'stimuli_features'] = features_by_stim
    # --- End Pre-computation ---

    # Collect all possible differing features across all trials
    all_differing_features = set()
    for _, row in participant_data.iterrows():
        stimuli_present = row['stimuli_present']
        stimuli_features = row['stimuli_features']

        # Identify differing dimensions between the stimuli
        differing_dimensions = []
        for dim in dimensions:
            dim_values = set(
                stimuli_features.get(stim_id, {}).get(dim, None)
                for stim_id in stimuli_present
            )
            dim_values.discard(None)
            if len(dim_values) > 1:
                differing_dimensions.append(dim)

        # Collect features present in the differing dimensions
        for dim in differing_dimensions:
            for stim_id in stimuli_present:
                feature_value = stimuli_features.get(stim_id, {}).get(dim, None)
                if feature_value is not None:
                    all_differing_features.add((dim, feature_value))

    all_differing_features = list(all_differing_features)
    num_all_differing_features = len(all_differing_features)

    # Initialize prior beliefs over all differing features
    prior_beliefs = {feature: 1.0 / num_all_differing_features for feature in all_differing_features}

    # Initialize columns & containers
    confidence_results = []
    participant_data['entropy'] = np.nan
    participant_data['BLR_confidence'] = np.nan
    participant_data['sum_prior_chosen_features'] = np.nan
    participant_data['norm_sum_prior_chosen_features'] = np.nan
    participant_data['posterior_sum_chosen_features'] = np.nan
    participant_data['norm_posterior_sum_chosen_features'] = np.nan
    participant_data['prior_beliefs'] = participant_data.get('prior_beliefs', '')
    participant_data['posterior_beliefs'] = participant_data.get('posterior_beliefs', '')
    participant_data['prior_beliefs'] = participant_data['prior_beliefs'].astype('object')
    participant_data['posterior_beliefs'] = participant_data['posterior_beliefs'].astype('object')

    for index, row in participant_data.iterrows():
        trial_overall = index + 1
        correctness = int(row['chosen_outcome'])  # 1 correct, 0 incorrect
        chosen_stimulus_num = int(row['response'])
        chosen_stimulus_id = f'stim{chosen_stimulus_num}'

        # Initialization metrics (before first update)
        if index == 0:
            init_entropy, init_BLR = compute_entropy(prior_beliefs)
            init_sum_prior = np.sum(list(prior_beliefs.values())) * 100
            confidence_results.append({
                'trial': trial_overall,
                'entropy': init_entropy,
                'BLR_confidence': init_BLR,
                'sum_prior_chosen_features': init_sum_prior,
                'chosen_outcome': None,
                'participant_confidence': None,
                'task_id': None,
                'shifts': 0
            })

        # Use precomputed stimuli present/features
        stimuli_present = row['stimuli_present']
        stimuli_features = row['stimuli_features']

        # Identify differing dimensions for THIS trial
        differing_dimensions = []
        for dim in dimensions:
            dim_values = set(
                stimuli_features.get(stim_id, {}).get(dim, None)
                for stim_id in stimuli_present
            )
            dim_values.discard(None)
            if len(dim_values) > 1:
                differing_dimensions.append(dim)

        # Extract chosen and unchosen features for differing dimensions
        chosen_features = stimuli_features.get(chosen_stimulus_id, {})
        unchosen_stimuli_features = {
            stim_id: stimuli_features.get(stim_id, {})
            for stim_id in stimuli_present if stim_id != chosen_stimulus_id
        }

        # Bayesian update
        posterior_beliefs = bayesian_update(
            prior_beliefs=prior_beliefs,
            chosen_features=chosen_features,
            unchosen_stimuli_features=unchosen_stimuli_features,
            feedback=correctness,
            differing_dimensions=differing_dimensions,
            task_id=task_id_tested
        )

        # Entropy / GC
        entropy, BLR_confidence = compute_entropy(posterior_beliefs)

        # Sum of posterior for chosen features (differing dims only)
        chosen_feature_posteriors = [
            posterior_beliefs.get((dim, chosen_features.get(dim, None)), 0.0)
            for dim in differing_dimensions if chosen_features.get(dim, None) is not None
        ]
        sum_posterior_chosen_features = (np.sum(chosen_feature_posteriors) * 100
                                         if chosen_feature_posteriors else 0.0)
        norm_posterior_sum_chosen_features = sum_posterior_chosen_features / 100.0

        # Sum of prior for chosen features (differing dims only)
        chosen_feature_priors = [
            prior_beliefs.get((dim, chosen_features.get(dim, None)), 0.0)
            for dim in differing_dimensions if chosen_features.get(dim, None) is not None
        ]
        sum_prior_chosen_features = (np.sum(chosen_feature_priors) * 100
                                     if chosen_feature_priors else 0.0)
        norm_sum_prior_chosen_features = sum_prior_chosen_features / 100.0

        # Store computed metrics
        participant_data.at[index, 'entropy'] = entropy
        participant_data.at[index, 'BLR_confidence'] = BLR_confidence
        participant_data.at[index, 'sum_prior_chosen_features'] = sum_prior_chosen_features
        participant_data.at[index, 'norm_sum_prior_chosen_features'] = norm_sum_prior_chosen_features
        participant_data.at[index, 'posterior_sum_chosen_features'] = sum_posterior_chosen_features
        participant_data.at[index, 'norm_posterior_sum_chosen_features'] = norm_posterior_sum_chosen_features
        participant_data.loc[index, 'prior_beliefs'] = json.dumps({str(k): v for k, v in prior_beliefs.items()})
        participant_data.loc[index, 'posterior_beliefs'] = json.dumps({str(k): v for k, v in posterior_beliefs.items()})

        # For plotting/export
        confidence_results.append({
            'trial': trial_overall,
            'entropy': entropy,
            'BLR_confidence': BLR_confidence,
            'sum_prior_chosen_features': sum_prior_chosen_features,
            'norm_sum_prior_chosen_features': norm_sum_prior_chosen_features,
            'posterior_sum_chosen_features': sum_posterior_chosen_features,
            'norm_posterior_sum_chosen_features': norm_posterior_sum_chosen_features,
            'chosen_outcome': correctness,
            'participant_confidence': row.get('confidence', None),
            'task_id': row.get('task_id', None),
            'IDs': row.get('IDs', 0),
            'EDs': row.get('EDs', 0),
            'shifts': row.get('shifts', 0)
        })

        # Update prior for next trial
        prior_beliefs = posterior_beliefs.copy()

    # Plot
    results_df = pd.DataFrame(confidence_results)
    results_df['posterior_sum_chosen_features_shifted'] = results_df['posterior_sum_chosen_features'].shift(+1)

    return participant_data


Process all data

In [ ]:
CrypticCreatures_adapted = CrypticCreatures
# --- Processing of all participants/task levels---
unique_ids = CrypticCreatures_adapted['id'].unique()
task_ids = CrypticCreatures_adapted['task_id'].dropna().unique()

# Initialize result columns if missing
cols_to_add = {
    'entropy': np.nan,
    'BLR_confidence': np.nan,
    'sum_prior_chosen_features': np.nan,
    'norm_sum_prior_chosen_features': np.nan,
    'posterior_sum_chosen_features': np.nan,
    'norm_posterior_sum_chosen_features': np.nan,
    'prior_beliefs': '',   
    'posterior_beliefs': ''
}
for col, default_val in cols_to_add.items():
    if col not in CrypticCreatures_adapted.columns:
        dtype = 'object' if 'beliefs' in col else None
        CrypticCreatures_adapted[col] = pd.Series(default_val, index=CrypticCreatures_adapted.index, dtype=dtype)

# Loop through IDs and task_ids, run participant processing, and write results back
for id_tested in unique_ids:
    for task_id_tested in task_ids:
        # print(f"Processing participant {id_tested}, task {task_id_tested}...")

        # Process this (id, task) slice
        participant_data = process_participant(CrypticCreatures_adapted, id_tested, task_id_tested)

        # Find rows to update in the main table
        participant_indices = CrypticCreatures_adapted.index[
            (CrypticCreatures_adapted['id'] == id_tested) & (CrypticCreatures_adapted['task_id'] == task_id_tested)
        ]

        # Write computed values back to the main DataFrame
        CrypticCreatures_adapted.loc[participant_indices, 'entropy'] = participant_data['entropy'].values
        CrypticCreatures_adapted.loc[participant_indices, 'BLR_confidence'] = participant_data['BLR_confidence'].values
        CrypticCreatures_adapted.loc[participant_indices, 'sum_prior_chosen_features'] = participant_data['sum_prior_chosen_features'].values
        CrypticCreatures_adapted.loc[participant_indices, 'posterior_sum_chosen_features'] = participant_data['posterior_sum_chosen_features'].values
        CrypticCreatures_adapted.loc[participant_indices, 'prior_beliefs'] = participant_data['prior_beliefs'].values
        CrypticCreatures_adapted.loc[participant_indices, 'posterior_beliefs'] = participant_data['posterior_beliefs'].values

In [ ]:
# Create and shift some additional variables
# Sort the DataFrame by 'id', 'task_id', 'run', and 'trial' to ensure correct order
CrypticCreatures_adapted = CrypticCreatures_adapted.sort_values(by=['id', 'task_id', 'run', 'trial']).reset_index(drop=True)
CrypticCreatures_adapted['total_iq'] = CrypticCreatures_adapted['icar_totalscore']

# Insert variable indicating task and block breaks
CrypticCreatures_adapted['task_break'] = ((CrypticCreatures_adapted['trial'] == 1) & (CrypticCreatures_adapted['run'] == 1)).astype(int)
CrypticCreatures_adapted['block_break'] = ((CrypticCreatures_adapted['trial'] == 1) & (CrypticCreatures_adapted['run'] == 2)).astype(int)

# Initialize empty columns for global_trial and task_trial
CrypticCreatures_adapted['global_trial'] = 0
CrypticCreatures_adapted['task_trial'] = 0

# Assign global trials and task-specific trials
for participant_id in CrypticCreatures_adapted['id'].unique():
    participant_data = CrypticCreatures_adapted[CrypticCreatures_adapted['id'] == participant_id]
    
    # Global trial counter (across all tasks and runs for this participant)
    global_trial_counter = 1
    
    for task in participant_data['task_id'].unique():
        task_data = participant_data[participant_data['task_id'] == task]
        task_trial_counter = 1  # Reset task-specific counter for each task_id
        
        for index in task_data.index:
            # Assign global_trial (continuous across all trials for the participant)
            CrypticCreatures_adapted.at[index, 'global_trial'] = global_trial_counter
            global_trial_counter += 1
            
            # Assign task_trial (continuous within each task_id across runs)
            CrypticCreatures_adapted.at[index, 'task_trial'] = task_trial_counter
            task_trial_counter += 1

# Display a preview
print(CrypticCreatures_adapted[['id', 'task_id', 'run', 'trial', 'global_trial', 'task_trial']].tail())


In [ ]:
# Ensure that 'id' is valid and there are no missing values in the ID column
CrypticCreatures_adapted = CrypticCreatures_adapted.dropna(subset=['id'])

# General shifts: where either IDs or EDs have a non-zero value
CrypticCreatures_adapted['general_shift'] = np.where((CrypticCreatures_adapted['IDs'] != 0) | (CrypticCreatures_adapted['EDs'] != 0), 1, 0)
    
# Create shifted variables with a backward shift (-1) per id and task_id
CrypticCreatures_adapted['prev_IDs'] = CrypticCreatures_adapted.groupby(['id','run', 'task_id'])['IDs'].shift(+1)
CrypticCreatures_adapted['prev_EDs'] = CrypticCreatures_adapted.groupby(['id','run', 'task_id'])['EDs'].shift(+1)
CrypticCreatures_adapted['prev_general_shift'] = CrypticCreatures_adapted.groupby(['id','run', 'task_id'])['general_shift'].shift(+1)
CrypticCreatures_adapted['prev_feedback'] = CrypticCreatures_adapted.groupby(['id','run', 'task_id'])['chosen_outcome'].shift(+1)
CrypticCreatures_adapted['prev_entropy'] = CrypticCreatures_adapted.groupby(['id','run', 'task_id'])['entropy'].shift(+1)
CrypticCreatures_adapted['prev_BLR_confidence'] = CrypticCreatures_adapted.groupby(['id','run', 'task_id'])['BLR_confidence'].shift(+1)
CrypticCreatures_adapted['prev_BLR_confidence_figure'] = CrypticCreatures_adapted.groupby(['id', 'run', 'task_id'])['BLR_confidence'].shift(1).fillna(0)
# Drop rows with missing values in the shifted columns and other relevant variables
CrypticCreatures_adapted_cleaned = CrypticCreatures_adapted.dropna(subset=['prev_IDs'])


In [ ]:
# Normalize to [0,1]
CrypticCreatures_adapted['norm_confidence'] = CrypticCreatures_adapted['confidence'] / 100
CrypticCreatures_adapted['norm_BLR_confidence'] = CrypticCreatures_adapted['BLR_confidence'] / 100
CrypticCreatures_adapted['norm_prev_BLR_confidence'] = CrypticCreatures_adapted['prev_BLR_confidence'] / 100
CrypticCreatures_adapted['norm_sum_prior_chosen_features'] = CrypticCreatures_adapted['sum_prior_chosen_features'] / 100

# Deviation columns (with safety checks)
required_cols = ['norm_confidence', 'norm_prev_BLR_confidence', 'norm_sum_prior_chosen_features']
if not set(required_cols).issubset(CrypticCreatures_adapted.columns):
    print("Warning: Required columns for deviation calculation missing. Skipping.")
    for col in ['confidence_deviation', 'prior_deviation', 'signed_confidence_deviation', 'signed_prior_deviation']:
        if col not in CrypticCreatures_adapted.columns:
            CrypticCreatures_adapted[col] = np.nan
else:
    # Signed deviations
    CrypticCreatures_adapted['signed_confidence_deviation'] = (
        CrypticCreatures_adapted['norm_confidence'] - CrypticCreatures_adapted['norm_prev_BLR_confidence']
    )
    CrypticCreatures_adapted['signed_prior_deviation'] = (
        CrypticCreatures_adapted['norm_confidence'] - CrypticCreatures_adapted['norm_sum_prior_chosen_features']
    )

    # Absolute deviations
    CrypticCreatures_adapted['confidence_deviation'] = CrypticCreatures_adapted['signed_confidence_deviation'].abs()
    CrypticCreatures_adapted['prior_deviation'] = CrypticCreatures_adapted['signed_prior_deviation'].abs()



Datasets relative to shifts etc.

In [ ]:
# Function to compute averages of Bayesian learner variables over relative trials
def compute_averages_per_person(shift_df, variables):
    """Compute average of variables for each person and trial relative to the shift.

    Args:
    - shift_df (DataFrame): The dataframe containing the windowed data around the shifts.
    - variables (list): List of variables (columns) to average.

    Returns:
    - averaged_df (DataFrame): The dataframe with averages per participant, shift type, and relative trial.
    """
    if shift_df.empty:
        print("Input shift_df is empty. Cannot compute averages.")
        return pd.DataFrame()

    # Define grouping columns
    grouping_cols = ['id', 'task_id', 'shift_type', 'nTrial_rel']
    # Add patientstatus if it exists
    if 'patientstatus' in shift_df.columns:
        grouping_cols.append('patientstatus')
    else:
        print("Warning: 'patientstatus' not found in shift data. Averaging without it.")

    # Check if all grouping columns exist
    if not all(col in shift_df.columns for col in grouping_cols):
        print("Warning: Not all grouping columns exist. Cannot compute averages.")
        return pd.DataFrame()

    # Filter variables to average to only those present in shift_df
    valid_variables = [var for var in variables if var in shift_df.columns]
    if not valid_variables:
        print("Warning: None of the specified variables to average exist in the dataframe.")
        return pd.DataFrame()
    print(f"Averaging variables: {valid_variables}")

    # Group by participant (id), task_id, shift_type, patientstatus, and relative trial, then compute the mean
    try:
        averaged_df = shift_df.groupby(grouping_cols)[valid_variables].mean().reset_index()
        return averaged_df
    except Exception as e:
         print(f"Error during grouping/averaging: {e}")
         return pd.DataFrame()



In [ ]:
def create_shift_dataframes(df, window=5): 
    """
    Create separate dataframes relative to all shifts (IDs, EDs, and general shifts), split by id and task_id.

    Args:
    - df (DataFrame): The dataset containing participant data.
    - window (int): Number of trials before and after the shift to consider.

    Returns:
    - merged_df (DataFrame): The final dataframe with all shifts data merged for all participants and tasks.
    """
     # Check required columns exist
    required_cols = ['id', 'task_id', 'trial', 'IDs', 'EDs', 'general_shift']
    if not all(col in df.columns for col in required_cols):
        print("Warning: Required columns for shift analysis missing. Returning empty DataFrame.")
        return pd.DataFrame()

    # Identify unique participants (id) and task ids
    participants = df['id'].unique()
    task_ids = df['task_id'].unique()

    # Initialize a list to hold all dataframes to be merged
    all_shift_data = []

    # Define a list of shift types
    shift_types = ['IDs', 'EDs', 'general_shift']

    # Loop over each participant
    for participant in participants:
        if pd.isna(participant): continue # Skip NaN IDs
        # Filter data for the current participant
        participant_data = df[df['id'] == participant]

        # Loop over each task_id for the current participant
        for task in task_ids:
            if pd.isna(task): continue #
            task_data = participant_data[participant_data['task_id'] == task]

            if task_data.empty: continue # Skip if no data for this combo

            # Loop over each shift type
            for shift_type in shift_types:
                # Check if shift_type column exists
                if shift_type not in task_data.columns:
                    print(f"Warning: Shift column '{shift_type}' not found. Skipping for this type.")
                    continue

                # Identify the shift trials
                shift_trials = task_data[task_data[shift_type].fillna(0) != 0]['trial']

                if shift_trials.empty: continue 

                # --- Define function to create a dataframe relative to shift trials ---
                def create_shift_df(shift_trials, shift_type, task_data, participant, task, window):
                    shift_list = []
                    for shift_trial in shift_trials:
                        window_data = task_data[(task_data['trial'] >= shift_trial - window) &
                                                (task_data['trial'] <= shift_trial + window)].copy() 
                        window_data['nTrial_rel'] = window_data['trial'] - shift_trial
                        window_data['shift_type'] = shift_type  
                        window_data['id'] = participant  
                        window_data['task_id'] = task  
                        shift_list.append(window_data)

                    if shift_list: 
                        shift_df = pd.concat(shift_list, ignore_index=True)
                        return shift_df
                    else:
                        return pd.DataFrame()

                
                shift_df = create_shift_df(shift_trials, shift_type, task_data, participant, task, window)

               
                if not shift_df.empty:
                    all_shift_data.append(shift_df)

    # Merge all data into one dataframe
    if all_shift_data: 
        merged_df = pd.concat(all_shift_data, ignore_index=True)
        return merged_df
    else:
        print("No shift data collected.")
        return pd.DataFrame() 



In [ ]:
# 1. Create the shift dataframes
merged_shift_df = create_shift_dataframes(CrypticCreatures_adapted)

# 2. Compute the averages per person and relative trial
variables_to_average = ['prev_BLR_confidence','prev_entropy','BLR_confidence','entropy','sum_prior_chosen_features','signed_confidence_deviation','signed_prior_deviation','confidence','chosen_outcome']  # Replace with actual variables you want to average
averaged_shift_df = compute_averages_per_person(merged_shift_df, variables_to_average)

ids_averaged_df = averaged_shift_df[averaged_shift_df['shift_type'] == 'IDs'].copy()
eds_averaged_df = averaged_shift_df[averaged_shift_df['shift_type'] == 'EDs'].copy()
generalshift_averaged_df = averaged_shift_df[averaged_shift_df['shift_type'] == 'general_shift'].copy()

CrypticCreaturesBayesianLearner_relativeShift_OCD = generalshift_averaged_df[generalshift_averaged_df['patientstatus'] == 1] 
CrypticCreaturesBayesianLearner_relativeShift_controls = generalshift_averaged_df[generalshift_averaged_df['patientstatus'] == 0] 

CrypticCreaturesBayesianLearner_ED_OCD = eds_averaged_df[eds_averaged_df['patientstatus'] == 1] 
CrypticCreaturesBayesianLearner_ED_controls = eds_averaged_df[eds_averaged_df['patientstatus'] == 0] 

CrypticCreaturesBayesianLearner_ID_OCD = ids_averaged_df[ids_averaged_df['patientstatus'] == 1] 
CrypticCreaturesBayesianLearner_ID_controls = ids_averaged_df[ids_averaged_df['patientstatus'] == 0] 


# Saving CrypticCreatures_adapted DataFrame as a CSV file
CrypticCreatures_adapted.to_csv('CrypticCreatures_BayesianLearner.csv', index=False)

ids_averaged_df.to_csv('CrypticCreaturesBayesianLearner_relativeID.csv', index=False)
eds_averaged_df.to_csv('CrypticCreaturesBayesianLearner_relativeED.csv', index=False)
generalshift_averaged_df.to_csv('CrypticCreaturesBayesianLearner_relativeShift.csv', index=False)

CrypticCreaturesBayesianLearner_ID_controls.to_csv('CrypticCreaturesBayesianLearner_relativeID_controls.csv', index=False)
CrypticCreaturesBayesianLearner_ID_OCD.to_csv('CrypticCreaturesBayesianLearner_relativeID_OCD.csv', index=False)

CrypticCreaturesBayesianLearner_ED_controls.to_csv('CrypticCreaturesBayesianLearner_relativeED_controls.csv', index=False)
CrypticCreaturesBayesianLearner_ED_OCD.to_csv('CrypticCreaturesBayesianLearner_relativeED_OCD.csv', index=False)

CrypticCreaturesBayesianLearner_relativeShift_controls.to_csv('CrypticCreaturesBayesianLearner_relativeShift_controls.csv', index=False)
CrypticCreaturesBayesianLearner_relativeShift_OCD.to_csv('CrypticCreaturesBayesianLearner_relativeShift_OCD.csv', index=False)
